# 11 - Fusion

Score-level fusion of the stylometric-neural detector and Binoculars (§3.3): a
logistic regression over the two scores, fitted on the tuning tenth of each
training set that neither base detector trained on.

Binoculars scored those tuning texts in a separate run
(`scripts/run_binoculars_tune.py`), since its original run covered only the
calibration and test splits.

1. Does fusion catch more machine text from unseen families at 1% FPR than
   either detector alone, on average and in the worst case? (RQ2, H2)
2. Which families does it help, and which does it not?

## 1. Load the scores

Both Binoculars files are needed: the original covers calibration and test, the
second covers the tuning slices.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from logos.paths import DATA_DIR
from logos.evaluation.metrics import (
    calibrate_threshold, tpr_at_threshold, fpr_at_threshold, auroc
)

MAGE_DIR = DATA_DIR / "mage"
INPUTS = ["stylometric_neural", "binoculars"]

meta = pd.read_parquet(
    MAGE_DIR / "corpus.parquet", columns=["text_id", "label", "family"]
)

stylometric = pd.concat(
    [pd.read_parquet(p)
     for p in sorted((MAGE_DIR / "scores" / "stylometric_neural").glob("*.parquet"))],
    ignore_index=True,
).rename(columns={"score": "stylometric_neural"})

binoculars = pd.concat([
    pd.read_parquet(MAGE_DIR / "scores" / "binoculars" / "all.parquet"),
    pd.read_parquet(MAGE_DIR / "scores" / "binoculars" / "tune.parquet"),
], ignore_index=True)[["text_id", "score"]].rename(columns={"score": "binoculars"})

scores = stylometric.merge(binoculars, on="text_id").merge(meta, on="text_id")

print(f"{scores.groupby(['training_set', 'seed']).ngroups} runs, {len(scores):,} rows")
print(scores.groupby("part").size().to_string())

40 runs, 3,732,102 rows
part
calibration    1136280
test           2099000
tune            496822


## 2. Fit and score all forty runs

The two scores are on very different scales, so they are standardised before
fitting. The scaler is fitted on the tuning slice alone, never on calibration or
test, for the same reason the base detectors' scaler is fitted on training rows
only (§3.3).

Each run is calibrated on its own validation humans, as in 10: forty different
fusions mean a shared threshold would sit at a different false-positive rate for
each.

The two fitted coefficients are kept. Binoculars is inverted on three families
(§4.2), so whether the fusion learns to down-weight it is the mechanism behind
whatever it does.

In [2]:
def fuse(run):
    tune = run[run["part"] == "tune"]
    calibration = run[run["part"] == "calibration"]
    test = run[run["part"] == "test"]

    scaler = StandardScaler().fit(tune[INPUTS])
    model = LogisticRegression().fit(scaler.transform(tune[INPUTS]), tune["label"])

    def fused(part):
        return model.decision_function(scaler.transform(part[INPUTS]))

    threshold = calibrate_threshold(
        fused(calibration), calibration["label"].to_numpy()
    )

    human = test[test["label"] == 0]
    human_scores = fused(human)

    rows = []
    for family, machine in test[test["label"] == 1].groupby("family"):
        values = np.r_[human_scores, fused(machine)]
        labels = np.r_[np.zeros(len(human)), np.ones(len(machine))]
        rows.append({
            "family": family,
            "tpr": tpr_at_threshold(values, labels, threshold),
            "auroc": auroc(values, labels),
            "test_fpr": fpr_at_threshold(values, labels, threshold),
            "w_stylometric_neural": model.coef_[0][0],
            "w_binoculars": model.coef_[0][1],
        })

    return pd.DataFrame(rows)


results = []
for (set_name, seed), run in scores.groupby(["training_set", "seed"]):
    out = fuse(run)
    out["training_set"], out["seed"] = set_name, seed
    results.append(out)

results = pd.concat(results, ignore_index=True)
results["condition"] = np.select(
    [results["training_set"] == "In-distribution",
     results["training_set"] == "LOFO " + results["family"]],
    ["In-distribution", "LOFO"],
    default="Other LOFO",
)

print("fitted weights, mean across runs:")
print(results[["w_stylometric_neural", "w_binoculars"]].mean().round(3).to_string())
print(f"\nFPR across all runs: {results['test_fpr'].min():.4f} to {results['test_fpr'].max():.4f}")

fitted weights, mean across runs:
w_stylometric    3.574
w_binoculars     0.157

FPR across all runs: 0.0083 to 0.0114


## 3. Results by family

In [3]:
def summarise(condition):
    g = results[results["condition"] == condition].groupby("family")
    return pd.DataFrame({
        f"tpr_{condition}": g["tpr"].mean(),
        f"sd_{condition}": g["tpr"].std(),
        f"auroc_{condition}": g["auroc"].mean(),
    })


by_family = summarise("In-distribution").join(summarise("LOFO"))
by_family["change"] = by_family["tpr_LOFO"] - by_family["tpr_In-distribution"]
by_family["two_sd"] = 2 * by_family[["sd_In-distribution", "sd_LOFO"]].max(axis=1)
by_family["interpretable"] = by_family["change"].abs() > by_family["two_sd"]

print(f"mean TPR    {by_family['tpr_In-distribution'].mean():.4f} -> {by_family['tpr_LOFO'].mean():.4f}")
print(f"mean AUROC  {by_family['auroc_In-distribution'].mean():.4f} -> {by_family['auroc_LOFO'].mean():.4f}")

by_family.sort_values("change").round(4)

mean TPR    0.6873 -> 0.5425
mean AUROC  0.9679 -> 0.9447


,tpr_In-distribution,sd_In-distribution,auroc_In-distribution,tpr_LOFO,sd_LOFO,auroc_LOFO,change,two_sd,interpretable
family,,,,,,,,,
OPT,0.8422,0.0261,0.9845,0.4535,0.0817,0.9516,-0.3887,0.1633,True
OpenAI GPT,0.7134,0.0123,0.9695,0.3802,0.0256,0.9261,-0.3332,0.0512,True
LLaMA,0.8145,0.0075,0.9811,0.6486,0.0867,0.9682,-0.1659,0.1735,False
FLAN-T5,0.1903,0.0360,0.9096,0.1360,0.0178,0.8510,-0.0543,0.0721,False
EleutherAI,0.9140,0.0352,0.9939,0.8630,0.0628,0.9927,-0.0510,0.1257,False
BigScience,0.4898,0.0357,0.9605,0.4687,0.0168,0.9488,-0.0211,0.0714,False
GLM-130B,0.8468,0.0061,0.9763,0.8472,0.0078,0.9744,0.0004,0.0156,False


## 4. Against the base detectors

H2 asks whether fusion beats both base detectors on average and in the worst
case, so both are reported. The comparison is fold by fold under shift (§3.5).
Binoculars has no seed variation, so the two-standard-deviation test against it
uses the fusion's spread alone.

In [4]:
lexical = pd.read_csv(DATA_DIR / "results" / "summary_lexical.csv")
binoculars = pd.read_csv(DATA_DIR / "results" / "summary_binoculars.csv")
stylometric = pd.read_csv(DATA_DIR / "results" / "summary_stylometric.csv")

comparison = pd.DataFrame({
    "lexical": lexical[lexical["condition"] == "LOFO"].set_index("family")["tpr_mean"],
    "binoculars": binoculars.set_index("family")["tpr_mean"],
    "stylometric_neural": stylometric[stylometric["condition"] == "LOFO"].set_index("family")["tpr_mean"],
    "fusion": by_family["tpr_LOFO"],
})
comparison["fusion_gain"] = comparison["fusion"] - comparison["stylometric_neural"]
comparison["clears_2sd"] = comparison["fusion_gain"].abs() > 2 * by_family["sd_LOFO"]

print(comparison[["lexical", "binoculars", "stylometric_neural", "fusion"]].mean().round(4).to_string())
print(f"\nworst case: stylometric {comparison['stylometric_neural'].min():.4f}, "
      f"fusion {comparison['fusion'].min():.4f}")

comparison.round(4).sort_values("fusion_gain")

lexical        0.1972
binoculars     0.1093
stylometric    0.5684
fusion         0.5425

worst case: stylometric 0.1118, fusion 0.1360


,lexical,binoculars,stylometric,fusion,fusion_gain,clears_2sd
family,,,,,,
OPT,0.2157,0.0004,0.6442,0.4535,-0.1907,True
EleutherAI,0.3985,0.0007,0.9402,0.8630,-0.0772,False
LLaMA,0.1025,0.1291,0.6625,0.6486,-0.0139,False
BigScience,0.2360,0.0178,0.4562,0.4687,0.0125,False
OpenAI GPT,0.1016,0.2918,0.3576,0.3802,0.0226,False
FLAN-T5,0.0704,0.0163,0.1118,0.1360,0.0242,False
GLM-130B,0.2555,0.3090,0.8065,0.8472,0.0407,True


## 5. Save

In [5]:
summary = results.groupby(["family", "condition"]).agg(
    tpr_mean=("tpr", "mean"), tpr_std=("tpr", "std"),
    auroc_mean=("auroc", "mean"), auroc_std=("auroc", "std"),
    test_fpr_mean=("test_fpr", "mean"), test_fpr_std=("test_fpr", "std"),
).reset_index()
summary.insert(0, "detector", "fusion")

out_path = DATA_DIR / "results" / "summary_fusion.csv"
summary.round(4).to_csv(out_path, index=False)

print(f"saved {len(summary)} rows to {out_path}")
summary.round(4)

saved 21 rows to /Users/xwm517/Desktop/logos/data/results/summary_fusion.csv


,detector,family,condition,tpr_mean,tpr_std,auroc_mean,auroc_std,test_fpr_mean,test_fpr_std
0,fusion,BigScience,In-distribution,0.4898,0.0357,0.9605,0.0019,0.0097,0.0003
1,fusion,BigScience,LOFO,0.4687,0.0168,0.9488,0.0019,0.0102,0.0005
2,fusion,BigScience,Other LOFO,0.5343,0.0823,0.9642,0.0077,0.0099,0.0006
3,fusion,EleutherAI,In-distribution,0.9140,0.0352,0.9939,0.0017,0.0097,0.0003
4,fusion,EleutherAI,LOFO,0.8630,0.0628,0.9927,0.0017,0.0099,0.0012
5,fusion,EleutherAI,Other LOFO,0.8465,0.1878,0.9922,0.0078,0.0099,0.0005
6,fusion,FLAN-T5,In-distribution,0.1903,0.0360,0.9096,0.0056,0.0097,0.0003
7,fusion,FLAN-T5,LOFO,0.1360,0.0178,0.8510,0.0038,0.0097,0.0004
8,fusion,FLAN-T5,Other LOFO,0.2558,0.0754,0.9218,0.0129,0.0099,0.0006
9,fusion,GLM-130B,In-distribution,0.8468,0.0061,0.9763,0.0015,0.0097,0.0003
